# A FAIR² Mental Health Survey Dataset from Kilifi County, Kenya: Demonstrating AI-Ready Data Standards for Data Infrastructure in Africa Exploration with `mlcroissant`

This notebook demonstrates how to explore, process, and analyze the FAIR² Mental Health Survey Dataset from Kilifi County, Kenya, using the [`mlcroissant`](https://pypi.org/project/mlcroissant/) library. 

The dataset contains survey data on mental health indicators among residents of Kilifi County, including demographic details and psychological assessment scores (GAD-7, PHQ-9, PSQ).

### Dataset Source
The dataset source is provided via a [Croissant schema](https://mlcommons.org/croissant/) URL.

In [ ]:
# Ensure `mlcroissant` library is installed
!pip install mlcroissant

## 1. Data Loading
Load metadata and records from the dataset using `mlcroissant`.

In [ ]:
import mlcroissant as mlc
import pandas as pd

# Define the dataset Croissant schema URL
croissant_url = "https://sen.science/doi/10.71728/senscience.vcs2-05nj/fair2.json"

# Load the Croissant dataset
dataset = mlc.Dataset(croissant_url)
meta = dataset.metadata   # This is an mlcroissant.DatasetMetadata object
# Display name and description
print(f"{meta.name}: {meta.description}\n")

## 2. Data Overview
Let’s review the available record sets, their `@id`s, and their field `@id`s using the `mlcroissant` API.
All references will use the `@id` as required for consistency.

This allows you to select exactly which data to load and which fields (columns) to analyze.

In [ ]:
# List all record sets by @id and their field @ids
print('Available record sets and fields:')
record_sets = dataset.record_sets
for rs in record_sets:
    print(f"RecordSet @id: {rs.id} (name: {rs.name})")
    for field in rs.fields:
        print(f"   Field @id: {field.id} (name: {field.name}, dataType: {field.data_type})")

## 3. Data Extraction
Let’s load all main record sets into Pandas DataFrames for downstream analysis.

You should use the `@id` from the previous overview section for precise extraction and for direct referencing in subsequent analysis steps.

In [ ]:
# Get the list of all record set @ids
record_set_ids = [rs.id for rs in dataset.record_sets]
print('Discovered record set @ids:', record_set_ids)

# For demonstration, we load the first record set; extend this to others similarly
dataframes = {}
for record_set_id in record_set_ids:
    print(f"Loading data for record set @id: {record_set_id}")
    records = list(dataset.records(record_set=record_set_id))
    if records:
        dataframes[record_set_id] = pd.DataFrame(records)
        print(f"Columns: {dataframes[record_set_id].columns.tolist()}")
        display(dataframes[record_set_id].head(2))
    else:
        print(f"No data rows found for record set {record_set_id}")

# If you know the main record set @id, you could alias it for convenience. Otherwise, see overview above.

## 4. Exploratory Data Analysis (EDA)
Let’s perform typical data processing steps: filtering records, normalizing a numeric field, and grouping by an attribute for aggregation.

> _**Be sure to use the correct `@id`s for fields and record set!**_
Below, we assume the main survey record set is `cr:main` (replace with the actual `@id` as per section 2 and your inspection), and that `PHQ-9_score` and `gender` are field names (check field list above to be certain and use the field `@id` as key).

In [ ]:
# Choose your actual record set @id (replace if needed)
main_record_set_id = record_set_ids[0]  # Assumes first record set is survey; adjust if necessary
df = dataframes[main_record_set_id]
# You should print df.columns to see actual field @ids (column names)
print('Fields / Columns in main DataFrame:', df.columns.tolist())

# Suppose PHQ-9 total score column is 'PHQ-9_score' (replace with actual field @id)
numeric_field_id = None
for name in df.columns:
    if 'phq' in name.lower() or 'phq9' in name.lower():
        numeric_field_id = name
        break

if numeric_field_id is None:
    print("Could not auto-locate PHQ-9 score field. Please check and set `numeric_field_id` to the correct field @id.")
else:
    print(f"Numeric field chosen for EDA: {numeric_field_id}")

    # Remove rows with missing values in numeric field
    filtered_df = df[df[numeric_field_id].notnull()]

    # Example threshold (e.g., scores > 10). Adjust as needed
    threshold = 10
    filtered_df = filtered_df[filtered_df[numeric_field_id] > threshold]
    print(f"Filtered records with {numeric_field_id} > {threshold}:")
    display(filtered_df.head())

    # Add a normalized score column
    normalized = (filtered_df[numeric_field_id] - filtered_df[numeric_field_id].mean()) / filtered_df[numeric_field_id].std()
    filtered_df[f"{numeric_field_id}_normalized"] = normalized
    print(f"Normalized {numeric_field_id} for filtered records:")
    display(filtered_df[[numeric_field_id, f"{numeric_field_id}_normalized"]].head())

    # Try to group by 'gender' (replace with correct group field @id as discovered above)
    group_field_id = None
    for name in df.columns:
        if 'gender' in name.lower():
            group_field_id = name
            break
    if group_field_id is not None:
        grouped_df = filtered_df.groupby(group_field_id)[numeric_field_id].agg(['mean', 'count'])
        print(f"Grouped data by {group_field_id} (mean and count):")
        display(grouped_df)
    else:
        print("No 'gender' field found to group by. Please check columns for available group fields.")

## 5. Visualization
Let’s plot distributions for numeric assessments. Modify the field `@id` as needed to match your actual column names.
- We use seaborn and matplotlib for histogram and boxplot visualizations.

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

# Plot PHQ-9 score distributions
if numeric_field_id is not None:
    plt.figure(figsize=(8,4))
    sns.histplot(df[numeric_field_id].dropna(), bins=20, kde=True)
    plt.title(f'Distribution of {numeric_field_id}')
    plt.xlabel(numeric_field_id)
    plt.show()

    # Optional: boxplot by gender group if available
    if group_field_id is not None:
        plt.figure(figsize=(6,4))
        sns.boxplot(x=group_field_id, y=numeric_field_id, data=df)
        plt.title(f'{numeric_field_id} by {group_field_id}')
        plt.show()

## 6. Conclusion

- We loaded the Croissant FAIR² mental health survey dataset using its schema and explored its metadata and tabular contents.
- We demonstrated how to access record sets, examine fields using their `@id`s, filter and normalize scores, perform group aggregations, and plot field distributions.
- This workflow enables FAIR, reproducible, and standards-compliant dataset analysis.

For deeper insights, extend the notebook by referencing additional record sets and fields using their `@id`s and exploring more advanced analyses or visualizations according to your research goals.